In [ ]:
MODELS = ["smollm2-360m"]
TASKS = ["goemotions", "ed_dialogue", "ed_situation"]
LIMIT = 40
TRAIN_LIMIT = 200
PRIMARY_MODEL = "qwen35-9b"
BATCH_SIZE = None
RERUN = False

In [ ]:
import gc, hashlib, json, math, os, random, re, string, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, files, userdata
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate", "pyyaml"], check=True)
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("no HF_TOKEN secret: downloads work, but more slowly")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from transformers import (AutoModelForCausalLM, AutoModelForImageTextToText, AutoModelForSequenceClassification,
                          AutoTokenizer, BitsAndBytesConfig, get_linear_schedule_with_warmup)

pd.set_option("display.max_colwidth", 120)
DRIVE_ROOT = Path("/content/drive/MyDrive/llm-persona-ei")


def find(name, candidates):
    path = next((path for path in candidates if path.exists()), None)
    if path is None and IN_COLAB:
        print(f"{name} not found: please upload it")
        path = Path(next(iter(files.upload())))
    assert path is not None, f"{name} not found in {[str(c) for c in candidates]}"
    return path


CONFIG_PATH = find("config.yaml", [Path("config.yaml"), Path("../config.yaml"), DRIVE_ROOT / "config.yaml"])
ROOT = Path(os.environ.get("PERSONA_EI_RESULTS") or (DRIVE_ROOT if IN_COLAB else CONFIG_PATH.resolve().parent))

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("config   :", CONFIG_PATH.resolve())
print("device   :", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
cfg = yaml.safe_load(CONFIG_PATH.read_text())
model_keys = cfg["tested"] if MODELS == "tested" else list(MODELS)
unknown = [key for key in model_keys if key not in cfg["models"]]
assert not unknown, f"not in config.yaml: {unknown}"
study_run = set(model_keys) <= set(cfg["tested"])

SEED = cfg["seed"]
BATCH_SIZE = BATCH_SIZE or cfg["engine"]["batch_size"]
CHUNK = cfg["engine"]["chunk_size"]
N_BOOTSTRAP = cfg["stats"]["n_bootstrap"]
CLASSIFY = cfg["decoding"]["classify"]
spec = cfg["rq2"]
BASELINE = "baseline-roberta"
BASELINE_SPEC = {**spec["baseline"], **({"train_limit": TRAIN_LIMIT} if TRAIN_LIMIT else {})}

OUT_DIR = ROOT / "rq2_emotion" / "results"
OUT_DIR = OUT_DIR if study_run else OUT_DIR / "small"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOADS = ROOT / "rq2_emotion" / "data" / "downloads"

print("models  :", ", ".join(model_keys), f"+ {BASELINE} (trained on {TRAIN_LIMIT or 'all'} texts per task)")
print("tasks   :", ", ".join(TASKS), f"({LIMIT or 'all'} test items each)")
print("results :", OUT_DIR, "" if study_run else "(not the study's models, so a folder of its own)")
pd.DataFrame([{"model": key, **{k: cfg["models"][key][k] for k in ("repo", "precision", "params_b")}} for key in model_keys])

In [ ]:
from datasets import load_dataset


def goemotions(split):
    dataset = load_dataset(spec["goemotions"]["repo"], "simplified", revision=spec["goemotions"]["revision"], split=split)
    names = dataset.features["labels"].feature.names
    table = dataset.to_pandas()
    table["labels"] = [[names[index] for index in indices] for indices in table["labels"]]
    return table[["id", "text", "labels"]], names


def empathetic_dialogues(split):
    archive = DOWNLOADS / "empatheticdialogues.tar.gz"
    if not archive.exists():
        DOWNLOADS.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(spec["empathetic_dialogues"]["url"], archive)
    with tarfile.open(archive) as tar:
        lines = tar.extractfile(f"empatheticdialogues/{split}.csv").read().decode("utf-8").splitlines()
    conversations = {}
    for line in lines[1:]:
        conv_id, _, emotion, situation, speaker, utterance = line.split(",")[:6]
        conversation = conversations.setdefault(conv_id, {"conv_id": conv_id, "emotion": emotion, "speaker": speaker,
                                                          "turns": [], "situation": situation.replace("_comma_", ",")})
        if speaker == conversation["speaker"]:
            conversation["turns"].append(utterance.replace("_comma_", ","))
    table = pd.DataFrame([{"conv_id": c["conv_id"], "emotion": c["emotion"], "situation": c["situation"],
                           "dialogue": "\n".join(c["turns"])} for c in conversations.values()])
    return table, sorted(table["emotion"].unique())


def load_tasks(split):
    go, go_labels = goemotions("validation" if split == "valid" else split)
    ed, ed_labels = empathetic_dialogues(split)
    every = {"goemotions": {"labels": go_labels, "max_labels": spec["goemotions"]["max_labels"],
                            "items": go.rename(columns={"id": "item_id", "labels": "gold"})}}
    for source in ("dialogue", "situation"):
        every[f"ed_{source}"] = {"labels": ed_labels, "max_labels": 1, "items": pd.DataFrame(
            {"item_id": ed["conv_id"], "text": ed[source], "gold": [[emotion] for emotion in ed["emotion"]]})}
    return {name: every[name] for name in TASKS}


test = load_tasks("test")
tasks = {name: {**task, "items": task["items"].head(LIMIT) if LIMIT else task["items"]} for name, task in test.items()}

go_train, go_labels = goemotions("train")
ed_train, _ = empathetic_dialogues("train")
mixed = go_train["labels"].str.len() >= 2
print(f"GoEmotions train: {len(go_train)} texts, {mixed.mean():.1%} with two or more emotions")
print("labels per text:", go_train["labels"].str.len().value_counts().sort_index().to_dict())
print(f"EmpatheticDialogues train: {len(ed_train)} conversations, {ed_train['emotion'].nunique()} emotions")
print("median words: GoEmotions text", go_train["text"].str.split().str.len().median(),
      "| ED speaker turns", ed_train["dialogue"].str.split().str.len().median(),
      "| ED situation", ed_train["situation"].str.split().str.len().median())
pairs = go_train.loc[mixed, "labels"].map(lambda labels: [" + ".join(sorted((a, b))) for i, a in enumerate(labels) for b in labels[i + 1:]])
print("most frequent pairs:", pairs.explode().value_counts().head(8).to_dict())
go_train["labels"].explode().value_counts().rename("texts").rename_axis("GoEmotions label (train)").to_frame().T

In [ ]:
SYSTEM = {
    "goemotions": ("You label the emotions expressed in short texts.\n"
                   "Choose between one and {max_labels} labels from this list: {labels}.\n"
                   'Answer with JSON only, for example: {{"emotions": ["joy"]}}'),
    "ed_dialogue": ("You name the emotion a person feels.\n"
                    "Choose exactly one label from this list: {labels}.\n"
                    'Answer with JSON only, for example: {{"emotions": ["proud"]}}'),
}
SYSTEM["ed_situation"] = SYSTEM["ed_dialogue"]
USER = {
    "goemotions": "Text: {text}",
    "ed_dialogue": "This is what the person said:\n{text}",
    "ed_situation": "The person describes this situation:\n{text}",
}

for name, task in tasks.items():
    item = task["items"].iloc[0]
    print(f"--- {name}: {len(task['items'])} items, {len(task['labels'])} labels")
    print("SYSTEM:", SYSTEM[name].format(labels=", ".join(task["labels"]), max_labels=task["max_labels"]))
    print("USER:  ", USER[name].format(text=item["text"]))
    print("CORRECT:", item["gold"], "\n")

In [ ]:
THINKING = re.compile(r"<think>.*?</think>", flags=re.DOTALL)


def load_model(key):
    spec = cfg["models"][key]
    tokenizer = AutoTokenizer.from_pretrained(spec["repo"], revision=spec["revision"], **spec.get("tokenizer_kwargs", {}))
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    options = {"revision": spec["revision"]}
    if DEVICE == "cuda":
        dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        options["device_map"] = "auto"
        if spec["precision"] in ("8bit", "4bit"):
            options["quantization_config"] = BitsAndBytesConfig(
                load_in_8bit=spec["precision"] == "8bit", load_in_4bit=spec["precision"] == "4bit",
                bnb_4bit_compute_dtype=dtype)
    else:
        if spec["precision"] != "bf16":
            raise RuntimeError(f"{key}: {spec['precision']} loading needs a CUDA GPU (run it on Colab)")
        dtype = torch.float32
    options["dtype"] = dtype

    print(f"   loading {key} ({spec['repo']}) ...", flush=True)
    try:
        model = AutoModelForCausalLM.from_pretrained(spec["repo"], **options)
    except ValueError:
        model = AutoModelForImageTextToText.from_pretrained(spec["repo"], **options)
    if DEVICE != "cuda":
        model = model.to(DEVICE)
    probe = tokenizer.apply_chat_template([{"role": "system", "content": "SYSTEM-PROBE"}, {"role": "user", "content": "USER-PROBE"}],
                                          tokenize=False, add_generation_prompt=True, **spec.get("chat_template_kwargs", {}))
    assert -1 < probe.find("SYSTEM-PROBE") < probe.find("USER-PROBE"), f"{key}: the chat template does not put the system prompt first"
    return tokenizer, model.eval()


def generate(tokenizer, model, key, conversations, decoding, seed=0):
    global BATCH_SIZE
    template_options = cfg["models"][key].get("chat_template_kwargs", {})
    prompts = [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **template_options)
               for messages in conversations]
    sampling = {"do_sample": True, "temperature": decoding["temperature"], "top_p": decoding["top_p"], "top_k": 0} \
        if decoding["temperature"] > 0 else {"do_sample": False}
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]), reverse=True)
    replies, start = [""] * len(prompts), 0
    while start < len(order):
        batch = order[start:start + BATCH_SIZE]
        encoded = tokenizer([prompts[i] for i in batch], return_tensors="pt", padding=True,
                            add_special_tokens=False).to(model.device)
        torch.manual_seed(seed + start)
        try:
            with torch.inference_mode():
                output = model.generate(**encoded, max_new_tokens=decoding["max_new_tokens"],
                                        pad_token_id=tokenizer.pad_token_id, repetition_penalty=1.0, **sampling)
        except torch.OutOfMemoryError:
            assert BATCH_SIZE > 1, "out of GPU memory with a batch size of 1"
            BATCH_SIZE = max(1, BATCH_SIZE // 2)
            torch.cuda.empty_cache()
            print(f"   out of GPU memory: batch size reduced to {BATCH_SIZE}", flush=True)
            continue
        texts = tokenizer.batch_decode(output[:, encoded["input_ids"].shape[1]:], skip_special_tokens=True)
        for index, text in zip(batch, texts):
            replies[index] = THINKING.sub("", text).strip()
        start += len(batch)
    return replies


def free():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def stable_hash(obj, n=8):
    text = json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:n]


def read_jsonl(path):
    rows = []
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return pd.DataFrame(rows)


def append_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    torn = False
    if path.exists() and path.stat().st_size:
        with open(path, "rb") as f:
            f.seek(-1, os.SEEK_END)
            torn = f.read(1) != b"\n"
    with open(path, "a", encoding="utf-8") as f:
        f.write(("\n" if torn else "") + "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows))


def parse_json(text):
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        value = json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None

In [ ]:
def jobs_for(key):
    revision = cfg["models"][key]["revision"]
    jobs = []
    for name, task in tasks.items():
        system = SYSTEM[name].format(labels=", ".join(task["labels"]), max_labels=task["max_labels"])
        for item in task["items"].itertuples():
            user = USER[name].format(text=item.text)
            job_id = f"rq2|{key}|{name}|{item.item_id}|" + stable_hash([system, user, CLASSIFY, revision])
            jobs.append({"key": f"{job_id}|t01", "job_id": job_id, "task": name, "item_id": item.item_id,
                         "system": system, "user": user})
    return jobs


for key in model_keys:
    path = OUT_DIR / "generations" / f"{key}.jsonl"
    if RERUN and path.exists():
        path.unlink()
    jobs = jobs_for(key)
    stored = read_jsonl(path)
    done = set(stored["key"]) if len(stored) else set()
    todo = [job for job in jobs if job["key"] not in done]
    print(f"{key}: {len(jobs)} items, {len(jobs) - len(todo)} stored, {len(todo)} to label", flush=True)
    if not todo:
        continue
    started = time.time()
    tokenizer, model = load_model(key)
    for start in range(0, len(todo), CHUNK):
        chunk = todo[start:start + CHUNK]
        texts = generate(tokenizer, model, key, [[{"role": "system", "content": job["system"]}, {"role": "user", "content": job["user"]}]
                                                 for job in chunk], CLASSIFY, int(stable_hash([chunk[0]["job_id"], 1]), 16) % 2**31)
        append_jsonl(path, [{"key": job["key"], "job_id": job["job_id"], "model": key, "task": job["task"], "item_id": job["item_id"],
                             "turn": 1, "user": job["user"], "assistant": text} for job, text in zip(chunk, texts)])
        print(f"   {start + len(chunk)}/{len(todo)}", flush=True)
    del tokenizer, model
    free()
    minutes = (time.time() - started) / 60
    every = sum(len(task["items"]) for task in test.values())
    print(f"{key}: {len(todo)} items in {minutes:.1f} min"
          + (f"; about {minutes / len(todo) * every:.0f} min for all {every} test items" if LIMIT else ""), flush=True)

shown = read_jsonl(OUT_DIR / "generations" / f"{model_keys[0]}.jsonl")
shown = shown[shown["key"].isin({job["key"] for job in jobs_for(model_keys[0])}) & (shown["task"] == TASKS[0])]
shown.sample(n=min(4, len(shown)), random_state=SEED).merge(tasks[TASKS[0]]["items"], on="item_id")[["text", "gold", "assistant"]]

In [ ]:
def matrix(label_lists, labels):
    position = {label: index for index, label in enumerate(labels)}
    found = np.zeros((len(label_lists), len(labels)), dtype=bool)
    for row, names in enumerate(label_lists):
        for name in names:
            if name in position:
                found[row, position[name]] = True
    return found


def f1_per_label(gold, predicted, weights=None):
    weights = np.ones(len(gold)) if weights is None else weights
    both = weights @ (gold & predicted)
    total = weights @ gold + weights @ predicted
    return np.divide(2 * both, total, out=np.zeros(gold.shape[1]), where=total > 0)


def macro_f1(gold, predicted, weights=None):
    return float(f1_per_label(gold, predicted, weights).mean())


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(seed)


def fit_predict(texts, targets, predict_on, multi_label):
    set_seed(SEED)
    device = "cuda" if DEVICE == "cuda" else "cpu"
    half_precision = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8
    tokenizer = AutoTokenizer.from_pretrained(BASELINE_SPEC["repo"], revision=BASELINE_SPEC["revision"])
    model = AutoModelForSequenceClassification.from_pretrained(
        BASELINE_SPEC["repo"], revision=BASELINE_SPEC["revision"], num_labels=targets.shape[1],
        problem_type="multi_label_classification" if multi_label else "single_label_classification").to(device)

    def encode(batch):
        return tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=BASELINE_SPEC["max_length"]).to(device)

    size = BASELINE_SPEC["batch_size"]
    steps = BASELINE_SPEC["epochs"] * math.ceil(len(texts) / size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASELINE_SPEC["learning_rate"], weight_decay=0.01)
    schedule = get_linear_schedule_with_warmup(optimizer, int(0.06 * steps), steps)
    rng = np.random.default_rng(SEED)
    model.train()
    for _ in range(BASELINE_SPEC["epochs"]):
        order = rng.permutation(len(texts))
        for start in range(0, len(order), size):
            index = order[start:start + size]
            target = torch.tensor(targets[index], device=device)
            target = target.float() if multi_label else target.float().argmax(dim=1)
            with torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=half_precision):
                loss = model(**encode([texts[i] for i in index]), labels=target).loss
            loss.backward()
            optimizer.step()
            schedule.step()
            optimizer.zero_grad()

    model.eval()
    results = []
    for batch_texts in predict_on:
        blocks = []
        for start in range(0, len(batch_texts), 4 * size):
            with torch.inference_mode():
                logits = model(**encode(batch_texts[start:start + 4 * size])).logits.float()
            blocks.append((torch.sigmoid(logits) if multi_label else torch.softmax(logits, dim=-1)).cpu().numpy())
        results.append(np.concatenate(blocks))
    del model
    free()
    return results


baseline_path = OUT_DIR / "generations" / f"{BASELINE}.jsonl"
if RERUN and baseline_path.exists():
    baseline_path.unlink()
stored = read_jsonl(baseline_path)
done = set(stored["key"]) if len(stored) else set()
baseline_key = lambda name, item_id: f"rq2|{BASELINE}|{name}|{item_id}|{stable_hash(BASELINE_SPEC)}"
train = valid = None
for name, task in tasks.items():
    if all(baseline_key(name, item_id) in done for item_id in task["items"]["item_id"]):
        print(f"{BASELINE} {name}: {len(task['items'])} predictions stored", flush=True)
        continue
    if train is None:
        train, valid = load_tasks("train"), load_tasks("valid")
    train_items, valid_items = (split[name]["items"].head(TRAIN_LIMIT) if TRAIN_LIMIT else split[name]["items"] for split in (train, valid))
    predict_items = test[name]["items"] if TRAIN_LIMIT is None else task["items"]
    print(f"{BASELINE} {name}: training on {len(train_items)} texts, predicting {len(predict_items)} ...", flush=True)
    labels, multi_label = task["labels"], name == "goemotions"
    valid_scores, test_scores = fit_predict(train_items["text"].tolist(), matrix(train_items["gold"], labels),
                                            [valid_items["text"].tolist(), predict_items["text"].tolist()], multi_label)
    threshold = 0.5
    if multi_label:
        gold = matrix(valid_items["gold"], labels)
        threshold = max(np.arange(0.1, 0.9, 0.05), key=lambda t: macro_f1(gold, valid_scores >= t))
    rows = []
    for item_id, scores in zip(predict_items["item_id"], test_scores):
        chosen = [labels[i] for i in np.flatnonzero(scores >= threshold)] if multi_label else []
        chosen = chosen or [labels[int(scores.argmax())]]
        rows.append({"key": baseline_key(name, item_id), "task": name, "item_id": item_id, "model": BASELINE,
                     "assistant": json.dumps({"emotions": chosen}), "threshold": float(threshold)})
    append_jsonl(baseline_path, rows)

In [ ]:
WORD_FORMS = [
    ("admiration", "admiring"), ("amusement", "amused"), ("anger", "angry"), ("annoyance", "annoyed"),
    ("anticipation", "anticipating"), ("anxiety", "anxious"), ("apprehension", "apprehensive"),
    ("approval", "approving"), ("care", "caring"), ("confidence", "confident"), ("confusion", "confused"),
    ("content", "contentment", "contented"), ("curiosity", "curious"), ("desire", "desiring"),
    ("devastation", "devastated"), ("disappointment", "disappointed"), ("disapproval", "disapproving"),
    ("disgust", "disgusted"), ("embarrassment", "embarrassed"), ("excitement", "excited"),
    ("faith", "faithful"), ("fear", "afraid", "fearful"), ("fury", "furious"), ("gratitude", "grateful"),
    ("grief", "grieving"), ("guilt", "guilty"), ("hope", "hopeful"), ("jealousy", "jealous"),
    ("joy", "joyful"), ("loneliness", "lonely"), ("love", "loving"), ("nervousness", "nervous"),
    ("nostalgia", "nostalgic"), ("optimism", "optimistic"), ("pride", "proud"), ("realization", "realizing"),
    ("relief", "relieved"), ("remorse", "remorseful"), ("sadness", "sad"), ("shame", "ashamed"),
    ("surprise", "surprised"), ("terror", "terrified"), ("trust", "trusting"),
]
EVERYDAY = {"admiring", "anticipating", "approving", "care", "desiring", "disapproving", "faith", "grieving",
            "hope", "loving", "realizing", "trust"}
NEGATIONS = {"not", "no", "never", "without", "nor", "neither", "t"}
EKMAN_GROUPS = {
    "anger": ["anger", "annoyance", "disapproval"],
    "disgust": ["disgust"],
    "fear": ["fear", "nervousness"],
    "joy": ["joy", "amusement", "approval", "excitement", "gratitude", "love", "optimism",
            "relief", "pride", "admiration", "desire", "caring"],
    "sadness": ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral": ["neutral"],
}


def normalize(text):
    text = "".join(" " if char in string.punctuation else char for char in str(text).lower())
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def label_forms(labels, in_text=False):
    forms = {label: label for label in labels}
    for group in WORD_FORMS:
        present = [label for label in labels if label in group]
        if len(present) == 1:
            forms.update({form: present[0] for form in group if form not in labels and not (in_text and form in EVERYDAY)})
    return forms


def read_labels(reply, labels, max_labels):
    answer = parse_json(reply)
    named = next((answer[key] for key in ("emotions", "emotion", "labels", "label") if key in answer), None) if answer else None
    if isinstance(named, str):
        named = [named]
    if isinstance(named, list):
        forms = label_forms(labels)
        found = [label for label in dict.fromkeys(forms.get(normalize(label)) for label in named) if label]
        return (found[:max_labels], "json") if found else ([], "none")
    forms = label_forms(labels, in_text=True)
    words = normalize(reply).split()
    found = list(dict.fromkeys(forms[word] for index, word in enumerate(words)
                               if word in forms and not NEGATIONS & set(words[max(0, index - 3):index])))
    if not found or len(found) > max_labels + 2:
        return [], "none"
    return found[:max_labels], "text"


def scores(gold, predicted):
    both, either = (gold & predicted).sum(axis=1), (gold | predicted).sum(axis=1)
    total = gold.sum(axis=1) + predicted.sum(axis=1)
    return {"macro_f1": macro_f1(gold, predicted),
            "micro_f1": float(2 * both.sum() / max(1, total.sum())),
            "per_item_f1": float(np.mean(np.divide(2 * both, total, out=np.zeros(len(gold)), where=total > 0))),
            "jaccard": float(np.mean(np.divide(both, either, out=np.zeros(len(gold)), where=either > 0))),
            "exact_match": float((gold == predicted).all(axis=1).mean())}


wanted = {job["key"] for key in model_keys for job in jobs_for(key)}
wanted |= {baseline_key(name, item_id) for name, task in tasks.items() for item_id in task["items"]["item_id"]}
generations = pd.concat([read_jsonl(OUT_DIR / "generations" / f"{key}.jsonl") for key in [*model_keys, BASELINE]], ignore_index=True)
generations = generations[generations["key"].isin(wanted)].drop_duplicates("key", keep="last")

predictions, by_task, per_label, single_mixed = [], [], [], []
for name, task in tasks.items():
    labels = task["labels"]
    rows = generations[generations["task"] == name].merge(task["items"][["item_id", "gold"]], on="item_id")
    read = [read_labels(reply, labels, task["max_labels"]) for reply in rows["assistant"]]
    rows = rows.assign(predicted=[found for found, _ in read], read_from=[how for _, how in read])
    predictions.append(rows[["model", "task", "item_id", "gold", "predicted", "read_from"]])
    for model, part in rows.groupby("model"):
        gold, predicted = matrix(part["gold"], labels), matrix(part["predicted"], labels)
        json_only = matrix([found if how == "json" else [] for found, how in zip(part["predicted"], part["read_from"])], labels)
        by_task.append({"model": model, "task": name, "items": len(part), **scores(gold, predicted),
                        "macro_f1_json_only": macro_f1(gold, json_only), "answers_as_json": (part["read_from"] == "json").mean(),
                        "no_label_found": (part["read_from"] == "none").mean()})
        if name != "goemotions":
            continue
        groups = list(EKMAN_GROUPS)
        to_groups = lambda column: [[g for g in groups if set(EKMAN_GROUPS[g]) & set(x)] for x in column]
        by_task[-1]["macro_f1_ekman"] = macro_f1(matrix(to_groups(part["gold"]), groups), matrix(to_groups(part["predicted"]), groups))
        per_label += [{"model": model, "label": label, "f1": value, "gold_texts": int(gold[:, i].sum())}
                      for i, (label, value) in enumerate(zip(labels, f1_per_label(gold, predicted)))]
        several = gold.sum(axis=1) >= 2
        for kind, mask in (("single emotion", ~several), ("mixed emotions", several)):
            if mask.any():
                single_mixed.append({"model": model, "texts": kind, "items": int(mask.sum()), **scores(gold[mask], predicted[mask]),
                                     "names_two_or_more": float((predicted[mask].sum(axis=1) >= 2).mean())})

predictions = pd.concat(predictions, ignore_index=True)
tables = {"scores_by_model": pd.DataFrame(by_task), "per_label_f1": pd.DataFrame(per_label), "single_vs_mixed": pd.DataFrame(single_mixed)}
(OUT_DIR / "scores").mkdir(exist_ok=True)
predictions.to_json(OUT_DIR / "scores" / "predictions.jsonl", orient="records", lines=True, force_ascii=False)
print(tables["scores_by_model"][["model", "task", "items", "macro_f1", "macro_f1_json_only", "answers_as_json", "exact_match"]]
      .round(3).to_string(index=False))

In [ ]:
if len(tables["per_label_f1"]):
    per_label = tables["per_label_f1"]
    print(per_label.pivot(index="label", columns="model", values="f1").join(per_label.groupby("label")["gold_texts"].first())
          .sort_values("gold_texts", ascending=False).round(2).to_string())
one = predictions[(predictions["task"] == "goemotions") & (predictions["model"] == model_keys[-1])]
wrong = [(gold, named) for correct, labels in zip(one["gold"], one["predicted"]) for gold in correct for named in labels if named not in correct]
pd.DataFrame(wrong, columns=["correct", f"{model_keys[-1]} said"]).value_counts().head(15).rename("texts").reset_index()

In [ ]:
def mean_ci(values, clusters=None):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return {"mean": np.nan, "lo": np.nan, "hi": np.nan, "n": 0}
    codes = np.arange(len(values)) if clusters is None else pd.factorize(np.asarray(clusters))[0]
    sums, counts = np.bincount(codes, weights=values), np.bincount(codes)
    rng = np.random.default_rng(SEED)
    means = []
    for start in range(0, N_BOOTSTRAP, 200):
        draws = rng.integers(0, len(sums), size=(min(200, N_BOOTSTRAP - start), len(sums)))
        means.append(sums[draws].sum(axis=1) / counts[draws].sum(axis=1))
    lo, hi = np.percentile(np.concatenate(means), [2.5, 97.5])
    return {"mean": float(values.mean()), "lo": float(lo), "hi": float(hi), "n": int(len(values))}


def summarize(part, value, by, cluster=None):
    return pd.DataFrame([{**dict(zip(by, key if isinstance(key, tuple) else (key,))),
                          **mean_ci(group[value], group[cluster] if cluster else None)}
                         for key, group in part.groupby(by, sort=True)])


go = predictions[predictions["task"] == "goemotions"]
rows = []
if len(go):
    labels = tasks["goemotions"]["labels"]
    wide = go.pivot(index="item_id", columns="model", values="predicted").dropna()
    gold = matrix(go.drop_duplicates("item_id").set_index("item_id").loc[wide.index, "gold"], labels)
    matrices = {model: matrix(wide[model], labels) for model in wide.columns}
    rng = np.random.default_rng(SEED)
    draws = [np.bincount(rng.integers(0, len(wide), len(wide)), minlength=len(wide)) for _ in range(N_BOOTSTRAP)]
    resampled = {model: np.array([macro_f1(gold, found, w) for w in draws]) for model, found in matrices.items()}
    for model, found in matrices.items():
        lo, hi = np.percentile(resampled[model], [2.5, 97.5])
        rows.append({"result": "macro-F1 on GoEmotions", "model": model, "mean": macro_f1(gold, found),
                     "lo": lo, "hi": hi, "items": len(wide), "p_value": np.nan})
    primary = PRIMARY_MODEL if PRIMARY_MODEL in matrices else next((model for model in matrices if model != BASELINE), None)
    if primary and BASELINE in matrices:
        difference = resampled[primary] - resampled[BASELINE]
        lo, hi = np.percentile(difference, [2.5, 97.5])
        rows.insert(0, {"result": f"MAIN TEST: {primary} minus {BASELINE} (paired bootstrap)", "model": primary,
                        "mean": macro_f1(gold, matrices[primary]) - macro_f1(gold, matrices[BASELINE]), "lo": lo, "hi": hi,
                        "items": len(wide), "p_value": min(1.0, 2 * min((difference <= 0).mean(), (difference >= 0).mean()))})
tables["primary_result"] = pd.DataFrame(rows, columns=["result", "model", "mean", "lo", "hi", "items", "p_value"])
print("MAIN RESULT (RQ2)\n" + tables["primary_result"].round(4).to_string(index=False))

by_model = [tables["primary_result"][tables["primary_result"]["result"] == "macro-F1 on GoEmotions"]
            .rename(columns={"items": "n"}).assign(measure="macro-F1 on GoEmotions")]
for task, source in (("ed_dialogue", "the speaker's turns"), ("ed_situation", "the situation")):
    part = predictions[predictions["task"] == task]
    if len(part):
        part = part.assign(correct=[float(gold == predicted) for gold, predicted in zip(part["gold"], part["predicted"])])
        by_model.append(summarize(part, "correct", ["model"]).assign(measure=f"accuracy on EmpatheticDialogues, from {source}"))
tables["by_model"] = pd.concat(by_model, ignore_index=True)[["measure", "model", "mean", "lo", "hi", "n"]]
ed = tables["scores_by_model"][tables["scores_by_model"]["task"] != "goemotions"]
if len(ed):
    tables["empathetic_dialogues"] = ed[["model", "task", "items", "exact_match", "macro_f1"]].rename(columns={"exact_match": "accuracy"})
    print("\nEMPATHETICDIALOGUES\n" + tables["empathetic_dialogues"].round(3).to_string(index=False))
if len(tables["single_vs_mixed"]):
    print("\nSINGLE VERSUS MIXED EMOTIONS (GoEmotions)\n" + tables["single_vs_mixed"].round(3).to_string(index=False))

In [ ]:
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
TEXT, MUTED, GRID, SURFACE = "#52514e", "#898781", "#e1e0d9", "#fcfcfb"


def bar_chart(part, x, y="mean", hue=None, title="", ylabel=""):
    categories = list(dict.fromkeys(part[x]))
    groups = list(dict.fromkeys(part[hue])) if hue else [None]
    figure, axis = plt.subplots(figsize=(max(6, 1.1 * len(categories) * max(1, len(groups) / 2)), 4), facecolor=SURFACE)
    width = (0.8 if hue else 0.5) / len(groups)
    for index, group in enumerate(groups):
        bars = (part if group is None else part[part[hue] == group]).set_index(x).reindex(categories)
        errors = np.vstack([bars[y] - bars["lo"], bars["hi"] - bars[y]]).clip(min=0) if {"lo", "hi"} <= set(part.columns) else None
        axis.bar(np.arange(len(categories)) + index * width, bars[y], width, yerr=errors, capsize=3,
                 color=SERIES[index % len(SERIES)], edgecolor=SURFACE, linewidth=2, ecolor=TEXT,
                 error_kw={"elinewidth": 1}, label=None if group is None else str(group))
    axis.set_xticks(np.arange(len(categories)) + width * (len(groups) - 1) / 2)
    axis.set_xticklabels(categories, rotation=25, ha="right")
    axis.set_xlim(-0.6, max(len(categories), 3) - 0.4 + width * (len(groups) - 1))
    axis.set_ylabel(ylabel or y, color=TEXT)
    axis.set_title(title, fontsize=11, loc="left", color=TEXT)
    axis.set_facecolor(SURFACE)
    axis.grid(axis="y", color=GRID, linewidth=1)
    axis.set_axisbelow(True)
    axis.tick_params(colors=MUTED, labelsize=8, length=0)
    for side in ("top", "right", "left"):
        axis.spines[side].set_visible(False)
    axis.spines["bottom"].set_color(GRID)
    if hue:
        axis.legend(title=hue, fontsize=8, title_fontsize=8, frameon=False, labelcolor=TEXT)
    return figure


figures = {}
f1 = tables["primary_result"][tables["primary_result"]["result"] == "macro-F1 on GoEmotions"]
if len(f1):
    figures["f1_by_model"] = bar_chart(f1, "model", title="GoEmotions: macro-F1 by model (95% interval)", ylabel="macro-F1")
if len(tables["single_vs_mixed"]):
    figures["single_vs_mixed"] = bar_chart(tables["single_vs_mixed"], "model", "per_item_f1", hue="texts",
                                           title="GoEmotions: single versus mixed emotions", ylabel="F1 per text")
if len(ed):
    figures["empathetic_dialogues"] = bar_chart(ed, "model", "exact_match", hue="task", ylabel="accuracy",
                                                title="EmpatheticDialogues: accuracy from the speaker's turns and from the situation")

In [ ]:
(OUT_DIR / "tables").mkdir(exist_ok=True)
(OUT_DIR / "figures").mkdir(exist_ok=True)
for name, content in tables.items():
    content.to_csv(OUT_DIR / "tables" / f"{name}.csv", index=False)
for name, figure in figures.items():
    figure.savefig(OUT_DIR / "figures" / f"{name}.png", dpi=130, bbox_inches="tight")
(OUT_DIR / "run_info.json").write_text(json.dumps({
    "models": model_keys, "tasks": TASKS, "limit": LIMIT, "primary_model": PRIMARY_MODEL, "decoding": CLASSIFY,
    "baseline": BASELINE_SPEC, "model_specs": {k: cfg["models"][k] for k in model_keys}, "device": DEVICE, "torch": torch.__version__,
}, indent=2))
print("saved to", OUT_DIR)
print("tables:", ", ".join(tables))